In this notebook, we will document and map every found issue with GDPR and AI act. 

# Notebook 03 — Data Governance & Compliance Audit
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook performs a structured compliance audit of the 500 cleaned credit application records produced by Notebook 01. Every data quality issue identified in the cleaning pipeline is mapped to its corresponding obligations under the **General Data Protection Regulation (GDPR)** and the **EU AI Act**, the two primary regulatory frameworks governing the collection, processing, and automated decision-making of personal data in the European Union. For each issue, this notebook documents the legal basis of the violation, assesses its severity, and provides concrete policy recommendations and remediation actions for NovaCred's Data Governance Officer.

### Regulatory Framework

**GDPR (Regulation 2016/679)** governs how personal data of EU individuals is collected, stored, processed, and protected. In the context of credit applications, virtually every field — name, email, SSN, date of birth, income — constitutes personal data and is subject to GDPR obligations including data minimisation, accuracy, storage limitation, and integrity.

**EU AI Act (Regulation 2024/1689)** classifies credit scoring as a **high-risk AI application** under Annex III. This means NovaCred is legally required to ensure the quality of data used to train and operate its credit scoring model, maintain human oversight over automated decisions, and keep detailed documentation of data quality issues and how they were resolved.

### Scope of This Audit

This audit covers the 14 data quality issues catalogued in Notebook 01, organised across five compliance dimensions:

| Dimension | Issues Covered |
|-----------|---------------|
| Uniqueness | Duplicate records, conflicting SSNs |
| Consistency | Gender coding, date formats |
| Validity | Income type, negative credit history, DTI ratio |
| Completeness | Missing income, DOB, email, SSN, timestamp |
| Accuracy | Age plausibility, timestamp plausibility, cross-field plausibility |

> **Note** — This audit uses the cleaned dataset `cleaned_credit_applications.csv` produced by Notebook 01 as its primary input. All flags and audit trail columns created during cleaning are used directly as evidence in this compliance analysis.

In [15]:
import pandas as pd
import regex as re

df_raw = pd.read_csv('../data/raw_credit_applications.csv')
df_clean = pd.read_csv('../data/cleaned_credit_applications.csv')

## Section 1 — Uniqueness

### #1 Duplicate Records & #2 Conflicting SSNs

In [9]:
# ─── #1: Duplicate _id records ───────────────────────────────────────────────
print('── #1 Duplicate Records ──')
print()
print('Raw dataset:')
dup_raw = df_raw[df_raw['_id'].duplicated(keep=False)][['_id', 'full_name', 'email', 'ssn']]
print(dup_raw.to_string(index=False))
print()
print('After cleaning:')
dup_clean = df_clean[df_clean['_id'].duplicated(keep=False)][['_id', 'full_name', 'email', 'ssn']]
if len(dup_clean) == 0:
    print('  No duplicate _id records found ✓')
print()

print('Affected rows in df_raw:')
print(dup_ids.to_string(index=False))
print()

print(f'Records in df_raw  : {len(df_raw)}')
print(f'Records in df_clean: {len(df_clean)}')
print(f'Removed            : {len(df_raw) - len(df_clean)} duplicate rows')


# ─── #2: Conflicting SSNs ────────────────────────────────────────────────────
print()
print('── #2 Conflicting SSNs ──')
print()

# Find SSNs that appear more than once
dup_ssns = df_raw['ssn'].value_counts()
dup_ssns = dup_ssns[dup_ssns > 1].index.tolist()

print('BEFORE (df_raw):')
print(df_raw[df_raw['ssn'].isin(dup_ssns)][['_id', 'full_name', 'ssn']].to_string(index=False))
print()
print('AFTER (df_clean):')
print(df_clean[df_clean['ssn'].isin(dup_ssns)][['_id', 'full_name', 'ssn']].to_string(index=False))
print()
print('  Classification:')
for ssn in dup_ssns:
    rows  = df_raw[df_raw['ssn'] == ssn]
    ids   = rows['_id'].tolist()
    names = rows['full_name'].tolist()
    if len(set(ids)) == 1:
        print(f'  {ssn} → Case A: exact duplicate row, handled by dedup')
    elif len(set(names)) == 1:
        print(f'  {ssn} → Case B: same person, two applications — flag for review')
    else:
        print(f'  {ssn} → Case C: different people, same SSN — escalate to legal')


── #1 Duplicate Records ──

Raw dataset:
    _id        full_name                       email         ssn
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com 427-90-1892
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com         NaN

After cleaning:
  No duplicate _id records found ✓

Affected rows in df_raw:
    _id        full_name                       email         ssn
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_042     Joseph Lopez     joseph.lopez1@gmail.com 652-70-5530
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com 427-90-1892
app_001 Stephanie Nguyen stephanie.nguyen47@mail.com         NaN

Records in df_raw  : 502
Records in df_clean: 500
Removed            : 2 duplicate rows

── #2 Conflicting SSNs ──

BEFORE (df_raw):
    _id      full_name         ssn
app_042   Joseph Lopez 652-70-5530
app_101   Sandra Smith 937-

### Compliance Analysis — Uniqueness

#### #1 — Duplicate Records

**GDPR — Article 5(1)(d) — Accuracy**
Personal data must be accurate and kept up to date. Duplicate records mean the same individual appears twice in the system, which can lead to incorrect decisions being made on stale or redundant data.

**GDPR — Article 5(1)(c) — Data Minimisation**
Data must be adequate, relevant, and limited to what is necessary. Retaining duplicate records violates this principle — NovaCred should not hold more records about an individual than necessary.

**Recommended Actions:**
- **Data Engineering Team**: Implement a unique constraint on `_id` at the point of data ingestion — reject any application whose ID already exists in the system.
- **Data Engineering Team**: Schedule periodic deduplication audits on the live database to catch any duplicates that slip through.

---

#### #2 — Conflicting SSNs

**Case A — Exact duplicate row (same `_id`, same name)**
This is purely a data pipeline issue, not an SSN conflict. The SSN itself is fine.
- **GDPR — Article 5(1)(c) — Data Minimisation**: redundant records must not be retained.
- **Action**: already resolved by deduplication in Notebook 01. No further action needed on the SSN.

**Case B — Same person, two applications (different `_id`, same name)**
A single person submitted two separate applications. This is not necessarily fraudulent but raises questions about whether both applications were processed independently.
- **GDPR — Article 5(1)(b) — Purpose Limitation**: data collected for one credit application should not be silently reused for a second without the applicant's knowledge.
- **Action**: Both records must be reviewed manually. If the second application was intentional, it should be processed normally. If it was a system error, one record should be removed and the applicant notified.

**Case C — Different people, same SSN (different `_id`, different name)**
Two different individuals claim the same Social Security Number. This is the most serious finding — it is either a data entry error or an indicator of identity fraud.
- **GDPR — Article 5(1)(d) — Accuracy**: at least one of these records contains incorrect personal data.
- **GDPR — Article 33 — Data Breach Notification**: if identity fraud is confirmed, NovaCred is legally obligated to notify the relevant supervisory authority within 72 hours.
- **EU AI Act — Article 10 — Data Governance**: corrupted identity data must not be used in any credit scoring model.
- **Action**: Both records must be immediately excluded from model training and any automated credit decision. The case must be escalated to NovaCred's legal team and, if fraud is confirmed, reported to the relevant authority.

## Section 2 - Consistency

### #3 Gender Coding & #4 Date Formats

In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# Consistency: Gender Coding & Date Formats
# ═══════════════════════════════════════════════════════════════════════════

# ─── #3: Gender Coding ───────────────────────────────────────────────────────
print('── #3 Gender Coding ──')
print()
print('BEFORE (df_raw) — inconsistent values:')
print(df_raw['gender_raw'].fillna('NULL').value_counts().to_string())
print()
print('AFTER (df_clean) — standardised values:')
print(df_clean['gender'].value_counts(dropna=False).to_string())

# ─── #4: Date Formats ────────────────────────────────────────────────────────
print()
print('── #4 Date Formats ──')
print()

# Detect format of each raw date string on the fly
def detect_format(d):
    if not d:
        return 'MISSING'
    if re.match(r'^\d{4}-\d{2}-\d{2}$', str(d)): return 'YYYY-MM-DD'
    if re.match(r'^\d{4}/\d{2}/\d{2}$', str(d)): return 'YYYY/MM/DD'
    if re.match(r'^\d{2}/\d{2}/\d{4}$', str(d)): return 'DD/MM/YYYY or MM/DD/YYYY'
    return 'UNKNOWN'

df_raw['dob_format_temp'] = df_raw['date_of_birth_raw'].apply(detect_format)

print('BEFORE (df_raw) — multiple formats detected:')
print(df_raw['dob_format_temp'].value_counts().to_string())
print()
print('Examples of each format:')
for fmt in df_raw['dob_format_temp'].unique():
    example = df_raw[df_raw['dob_format_temp'] == fmt]['date_of_birth_raw'].iloc[0]
    print(f'  {fmt:<30} → e.g. "{example}"')

print()
print('AFTER (df_clean) — all dates in ISO 8601:')
all_uniform = df_clean['date_of_birth'].dropna().apply(lambda x: str(x)[:10]).str.match(r'^\d{4}-\d{2}-\d{2}$').all()
print(df_clean['date_of_birth'].dropna().apply(lambda x: str(x)[:10]).head(5).to_string())
print()
print(f'  All parsed dates follow YYYY-MM-DD format: {all_uniform} ✓')

# clean up temp column
df_raw.drop(columns=['dob_format_temp'], inplace=True)

── #3 Gender Coding ──

BEFORE (df_raw) — inconsistent values:
gender_raw
Male      195
Female    193
F          58
M          53
NULL        3

AFTER (df_clean) — standardised values:
gender
Female    250
Male      247
NaN         3

── #4 Date Formats ──

BEFORE (df_raw) — multiple formats detected:
dob_format_temp
YYYY-MM-DD                  340
DD/MM/YYYY or MM/DD/YYYY    101
YYYY/MM/DD                   56
UNKNOWN                       5

Examples of each format:
  YYYY-MM-DD                     → e.g. "2001-03-09"
  DD/MM/YYYY or MM/DD/YYYY       → e.g. "14/02/1982"
  YYYY/MM/DD                     → e.g. "1990/07/26"
  UNKNOWN                        → e.g. "nan"

AFTER (df_clean) — all dates in ISO 8601:
0    2001-03-09
1    1992-03-31
2    1989-10-24
3    1983-04-25
4    1999-05-21

  All parsed dates follow YYYY-MM-DD format: True ✓
